In [1]:
import socket

for host in ["pypi.org", "github.com", "api.github.com", "dataverse.no"]:
    try:
        socket.getaddrinfo(host, 443)
        print(host, "→ DNS OK")
    except socket.gaierror:
        print(host, "→ DNS FAILED")

pypi.org → DNS OK
github.com → DNS OK
api.github.com → DNS OK
dataverse.no → DNS OK


In [2]:
from pathlib import Path
import requests
import subprocess

output_dir = Path("/kaggle/working/ruod_download")
output_dir.mkdir(parents=True, exist_ok=True)

api_url = (
    "https://api.github.com/repos/xiaoDetection/RUOD/"
    "releases/tags/untagged-4f7c7ab75187d68b6449"
)

response = requests.get(api_url, timeout=30)
response.raise_for_status()
release = response.json()

assets = {
    asset["name"]: asset
    for asset in release.get("assets", [])
}

for name in ["RUOD.tar.partaa", "RUOD.tar.partab"]:
    if name not in assets:
        raise RuntimeError(
            f"{name} not found. Available files: {list(assets)}"
        )

    asset = assets[name]
    destination = output_dir / name
    expected_size = asset["size"]

    if (
        destination.exists()
        and destination.stat().st_size == expected_size
    ):
        print("Already downloaded:", name)
        continue

    print("Downloading:", name, flush=True)

    subprocess.run(
        [
            "curl",
            "--fail",
            "--location",
            "--retry", "3",
            "--connect-timeout", "30",
            "--continue-at", "-",
            "--output", str(destination),
            asset["browser_download_url"]
        ],
        check=True
    )

    if destination.stat().st_size != expected_size:
        raise RuntimeError(f"Incomplete download: {name}")

    print("Complete:", name, flush=True)

print("\nBoth RUOD parts downloaded successfully.")

Downloading: RUOD.tar.partaa


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
 99 2000M   99 1994M    0     0  11.8M      0  0:02:48  0:02:48 --:--:-- 13.8M

Complete: RUOD.tar.partaa
Downloading: RUOD.tar.partab


100 2000M  100 2000M    0     0  11.8M      0  0:02:48  0:02:48 --:--:-- 12.6M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1541M  100 1541M    0     0  16.4M      0  0:01:33  0:01:33 --:--:-- 12.7M


Complete: RUOD.tar.partab

Both RUOD parts downloaded successfully.


In [3]:
from pathlib import Path
from collections import Counter
import shutil
import tarfile

download_dir = Path("/kaggle/working/ruod_download")
parts = [
    download_dir / "RUOD.tar.partaa",
    download_dir / "RUOD.tar.partab"
]

archive_path = download_dir / "RUOD.tar"
output_dir = Path("/kaggle/working/ruod_original")

for part in parts:
    if not part.exists():
        raise FileNotFoundError(part)

# Join the parts in the required order.
expected_size = sum(part.stat().st_size for part in parts)

if not (
    archive_path.exists()
    and archive_path.stat().st_size == expected_size
):
    print("Joining archive parts...")

    temporary = download_dir / "RUOD.tar.partial"

    with temporary.open("wb") as destination:
        for part in parts:
            with part.open("rb") as source:
                shutil.copyfileobj(
                    source, destination,
                    length=8 * 1024 * 1024
                )

    temporary.replace(archive_path)

print("Extracting...")
output_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(archive_path, mode="r:*") as archive:
    archive.extractall(path=output_dir, filter="data")

files = sorted(
    path for path in output_dir.rglob("*") if path.is_file()
)
counts = Counter(path.suffix.lower() for path in files)

print("\nEXTRACTION COMPLETE")
print("Folder:", output_dir)

print("\nFILE COUNTS")
for extension, count in counts.most_common(10):
    print(extension or "(no extension)", ":", count)

print("\nANNOTATION / CONFIG EXAMPLES")
for extension in [".xml", ".json", ".txt", ".yaml", ".yml"]:
    examples = [
        str(path.relative_to(output_dir))
        for path in files
        if path.suffix.lower() == extension
    ][:3]

    if examples:
        print(extension, ":", examples)

Joining archive parts...
Extracting...

EXTRACTION COMPLETE
Folder: /kaggle/working/ruod_original

FILE COUNTS
.jpg : 14300
.json : 5

ANNOTATION / CONFIG EXAMPLES
.json : ['RUOD/Environmet_ANN/instances_blur.json', 'RUOD/Environmet_ANN/instances_color.json', 'RUOD/Environmet_ANN/instances_light.json']


In [4]:
from pathlib import Path
import subprocess
import hashlib

folder = Path("/kaggle/working/norfisk_download")
folder.mkdir(parents=True, exist_ok=True)

zip_path = folder / "NorFisk_v1.0.zip"

url = (
    "https://dataverse.no/api/access/datafile/:persistentId"
    "?persistentId=doi:10.18710/H5G3K5/QK355D"
)

expected_md5 = "6ec799e74054dec2dadb6f9179e51ea5"

def file_md5(path):
    digest = hashlib.md5()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if zip_path.exists() and file_md5(zip_path) == expected_md5:
    print("Verified ZIP already exists.")
else:
    subprocess.run([
        "curl",
        "--fail",
        "--location",
        "--retry", "3",
        "--connect-timeout", "30",
        "--continue-at", "-",
        "--output", str(zip_path),
        url
    ], check=True)

    actual_md5 = file_md5(zip_path)

    if actual_md5 != expected_md5:
        raise RuntimeError(
            f"Checksum mismatch. Do not extract. MD5: {actual_md5}"
        )

print("NorFisk ZIP verified successfully.")
print("File:", zip_path)
print("Size (GB):", round(zip_path.stat().st_size / 1024**3, 2))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 2518M  100 2518M    0     0  10.8M      0  0:03:52  0:03:52 --:--:-- 15.6M


NorFisk ZIP verified successfully.
File: /kaggle/working/norfisk_download/NorFisk_v1.0.zip
Size (GB): 2.46


## Checking norfisk

In [5]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter

zip_path = Path(
    "/kaggle/working/norfisk_download/NorFisk_v1.0.zip"
)
output_dir = Path("/kaggle/working/norfisk_original")

if not zip_path.exists():
    raise FileNotFoundError(
        "এই session-এ ZIP নেই। Saved output input হিসেবে "
        "যোগ করতে হবে—আবার download এখনই কোরো না।"
    )

output_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_path) as archive:
    # Check paths before extraction.
    for member in archive.infolist():
        destination = (output_dir / member.filename).resolve()
        if not destination.is_relative_to(output_dir.resolve()):
            raise ValueError(f"Unsafe archive path: {member.filename}")

    print("Extracting NorFisk...")
    archive.extractall(output_dir)

files = sorted(
    path for path in output_dir.rglob("*") if path.is_file()
)
counts = Counter(path.suffix.lower() for path in files)

print("\nFILE COUNTS")
for extension, count in counts.most_common(10):
    print(extension or "(no extension)", ":", count)

print("\nEXAMPLE PATHS")
for extension in [".jpg", ".jpeg", ".png", ".csv", ".txt", ".xml", ".json"]:
    examples = [
        str(path.relative_to(output_dir))
        for path in files if path.suffix.lower() == extension
    ][:3]
    if examples:
        print(extension, ":", examples)

# Inspect a few small text/CSV annotation candidates.
candidates = [
    path for path in files
    if path.suffix.lower() in {".csv", ".txt"}
    and "annot" in str(path).lower()
]

for path in candidates[:3]:
    print("\nANNOTATION SAMPLE:", path.relative_to(output_dir))
    with path.open(errors="replace") as file:
        for _ in range(3):
            line = file.readline()
            if not line:
                break
            print(line.rstrip()[:500])

Extracting NorFisk...

FILE COUNTS
.txt : 12514
.png : 12514

EXAMPLE PATHS
.png : ['saithe/img/saithe.000001.png', 'saithe/img/saithe.000002.png', 'saithe/img/saithe.000003.png']
.txt : ['saithe/annotations/saithe.000001.txt', 'saithe/annotations/saithe.000002.txt', 'saithe/annotations/saithe.000003.txt']

ANNOTATION SAMPLE: saithe/annotations/saithe.000001.txt
saithe.000001.png,saithe,65,41,170,137,235,177

ANNOTATION SAMPLE: saithe/annotations/saithe.000002.txt
saithe.000002.png,saithe,76,50,196,155,271,204

ANNOTATION SAMPLE: saithe/annotations/saithe.000003.txt
saithe.000003.png,saithe,73,56,216,163,288,218


In [6]:
## Compare Classes and Box Counts Across All Three Datasets

In [8]:
from pathlib import Path
from collections import Counter
import json
import csv
import pandas as pd

report_dir = Path("/kaggle/working/three_domain_audit")
report_dir.mkdir(parents=True, exist_ok=True)

class_rows = []
file_rows = []

def inspect_coco(dataset, path):
    data = json.loads(path.read_text())

    if not (
        isinstance(data, dict)
        and all(key in data for key in [
            "images", "annotations", "categories"
        ])
    ):
        return

    categories = {
        category["id"]: category["name"]
        for category in data["categories"]
    }
    counts = Counter(
        annotation["category_id"]
        for annotation in data["annotations"]
    )

    file_rows.append({
        "dataset": dataset,
        "annotation_file": str(path),
        "image_records": len(data["images"]),
        "boxes": len(data["annotations"])
    })

    for category_id, name in categories.items():
        class_rows.append({
            "dataset": dataset,
            "annotation_file": path.name,
            "class": name,
            "boxes": counts[category_id]
        })

# Brackish: read only ground-truth JSONs.
brackish_files = [
    path for path in Path("/kaggle/input").rglob("*groundtruth.json")
    if "brackish" in str(path).lower()
]

if not brackish_files:
    raise FileNotFoundError("Brackish dataset Input-এ add করো।")

for path in sorted(brackish_files):
    inspect_coco("Brackish", path)

# RUOD: keep each JSON separate to avoid double-counting subsets.
ruod_dir = Path("/kaggle/working/ruod_original")

if not ruod_dir.exists():
    raise FileNotFoundError("Extracted RUOD folder পাওয়া যায়নি।")

for path in sorted(ruod_dir.rglob("*.json")):
    inspect_coco("RUOD", path)

# NorFisk: filename,class,xmin,ymin,xmax,ymax,width,height.
norfisk_dir = Path("/kaggle/working/norfisk_original")
norfisk_counts = Counter()
norfisk_image_names = set()
annotation_count = 0

for path in sorted(norfisk_dir.rglob("*.txt")):
    if path.parent.name.lower() != "annotations":
        continue

    annotation_count += 1

    with path.open(newline="") as file:
        for row in csv.reader(file):
            if not row or not any(value.strip() for value in row):
                continue

            if len(row) != 8:
                raise ValueError(f"Unexpected annotation: {path}: {row}")

            # Verify that coordinate fields are numeric.
            [float(value) for value in row[2:]]

            norfisk_image_names.add(row[0].strip())
            norfisk_counts[row[1].strip()] += 1

for name, count in sorted(norfisk_counts.items()):
    class_rows.append({
        "dataset": "NorFisk",
        "annotation_file": "individual TXT files",
        "class": name,
        "boxes": count
    })

file_rows.append({
    "dataset": "NorFisk",
    "annotation_file": f"{annotation_count} TXT files",
    "image_records": len(norfisk_image_names),
    "boxes": sum(norfisk_counts.values())
})

classes = pd.DataFrame(class_rows)
inventory = pd.DataFrame(file_rows)

classes.to_csv(report_dir / "class_box_counts.csv", index=False)
inventory.to_csv(report_dir / "annotation_inventory.csv", index=False)

print("ANNOTATION FILE SUMMARY")
display(inventory)

print("\nBRACKISH — totals across original splits")
display(
    classes[classes["dataset"] == "Brackish"]
    .groupby("class", as_index=False)["boxes"].sum()
)

print("\nRUOD — each annotation file shown separately")
display(classes[classes["dataset"] == "RUOD"])

print("\nNORFISK")
display(classes[classes["dataset"] == "NorFisk"])

print("\nSaved reports:", report_dir)

ANNOTATION FILE SUMMARY


,dataset,annotation_file,image_records,boxes
0,Brackish,/kaggle/input/datasets/aalborguniversity/brack...,1239,3466
1,Brackish,/kaggle/input/datasets/aalborguniversity/brack...,9967,28518
2,Brackish,/kaggle/input/datasets/aalborguniversity/brack...,1238,3581
3,RUOD,/kaggle/working/ruod_original/RUOD/Environmet_...,100,690
4,RUOD,/kaggle/working/ruod_original/RUOD/Environmet_...,100,551
5,RUOD,/kaggle/working/ruod_original/RUOD/Environmet_...,100,487
6,RUOD,/kaggle/working/ruod_original/RUOD/RUOD_ANN/in...,4200,22969
7,RUOD,/kaggle/working/ruod_original/RUOD/RUOD_ANN/in...,9800,51935
8,NorFisk,12514 TXT files,12514,12514



BRACKISH — totals across original splits


,class,boxes
0,crab,12348
1,fish,3352
2,jellyfish,637
3,shrimp,548
4,small_fish,10768
5,starfish,7912



RUOD — each annotation file shown separately


,dataset,annotation_file,class,boxes
18,RUOD,instances_blur.json,holothurian,178
19,RUOD,instances_blur.json,echinus,186
20,RUOD,instances_blur.json,scallop,95
21,RUOD,instances_blur.json,starfish,76
22,RUOD,instances_blur.json,fish,48
23,RUOD,instances_blur.json,corals,32
24,RUOD,instances_blur.json,diver,43
25,RUOD,instances_blur.json,cuttlefish,5
26,RUOD,instances_blur.json,turtle,23
27,RUOD,instances_blur.json,jellyfish,4



NORFISK


,dataset,annotation_file,class,boxes
68,NorFisk,individual TXT files,saithe,3027
69,NorFisk,individual TXT files,salmonid,9487



Saved reports: /kaggle/working/three_domain_audit


## Verify NorFisk Images, Dimensions and Boxes

In [9]:
from pathlib import Path
from PIL import Image
import csv
import math
import pandas as pd

root = Path("/kaggle/working/norfisk_original")
output = Path("/kaggle/working/three_domain_audit")
output.mkdir(parents=True, exist_ok=True)

records = []
issues = []

for annotation in sorted(root.rglob("*.txt")):
    if annotation.parent.name.lower() != "annotations":
        continue

    with annotation.open(newline="") as file:
        for row in csv.reader(file):
            if not row or not any(value.strip() for value in row):
                continue

            if len(row) != 8:
                issues.append({
                    "file": str(annotation),
                    "issue": "unexpected_row_format"
                })
                continue

            name, label = row[0].strip(), row[1].strip()
            image_path = annotation.parent.parent / "img" / name

            if not image_path.exists():
                issues.append({
                    "file": str(annotation),
                    "issue": "missing_image"
                })
                continue

            try:
                x1, y1, x2, y2, aw, ah = map(float, row[2:])

                with Image.open(image_path) as image:
                    width, height = image.size

                problems = []

                if not all(math.isfinite(v) for v in [
                    x1, y1, x2, y2, aw, ah
                ]):
                    problems.append("non_finite")
                else:
                    if (width, height) != (aw, ah):
                        problems.append("dimension_mismatch")

                    if x2 <= x1 or y2 <= y1:
                        problems.append("non_positive_box")

                    if (
                        x1 < 0 or y1 < 0
                        or x2 > width or y2 > height
                    ):
                        problems.append("out_of_bounds")

                for problem in problems:
                    issues.append({
                        "file": str(annotation),
                        "issue": problem
                    })

                records.append({
                    "image_path": str(image_path),
                    "annotation_path": str(annotation),
                    "original_class": label,
                    "width": width,
                    "height": height,
                    "xmin": x1, "ymin": y1,
                    "xmax": x2, "ymax": y2,
                    "status": "ok" if not problems else "|".join(problems)
                })

            except Exception as error:
                issues.append({
                    "file": str(annotation),
                    "issue": f"read_error: {error}"
                })

manifest = pd.DataFrame(records)
issue_table = pd.DataFrame(issues, columns=["file", "issue"])

manifest.to_csv(output / "norfisk_manifest.csv", index=False)
issue_table.to_csv(output / "norfisk_issues.csv", index=False)

print("Annotation rows matched to readable images:", len(manifest))
print("Unique matched images:", manifest["image_path"].nunique())
print("Issue entries:", len(issue_table))

if len(issue_table):
    display(issue_table["issue"].value_counts())
else:
    print("Image matching, dimensions and box boundaries: PASSED")

display(
    manifest.groupby("original_class").agg(
        images=("image_path", "nunique"),
        min_width=("width", "min"),
        max_width=("width", "max"),
        min_height=("height", "min"),
        max_height=("height", "max")
    )
)

print("Reports saved:", output)

Annotation rows matched to readable images: 12514
Unique matched images: 12514
Issue entries: 0
Image matching, dimensions and box boundaries: PASSED


,images,min_width,max_width,min_height,max_height
original_class,,,,,
saithe,3027,125,3190,94,2156
salmonid,9487,101,1916,77,1295


Reports saved: /kaggle/working/three_domain_audit


## Match RUOD Images and Convert Fish Boxes to YOLO

In [10]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import json
import pandas as pd

source = Path("/kaggle/working/ruod_original")
audit = Path("/kaggle/working/three_domain_audit")
audit.mkdir(parents=True, exist_ok=True)

lookup = defaultdict(list)

for path in source.rglob("*"):
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup[path.name].append(path)

issues = []
checked = 0

for split in ["train", "test"]:
    matches = [
        path for path in source.rglob(f"instances_{split}.json")
        if path.parent.name == "RUOD_ANN"
    ]

    if len(matches) != 1:
        raise RuntimeError(f"Cannot locate {split} annotation uniquely")

    data = json.loads(matches[0].read_text())

    for record in data["images"]:
        name = Path(record["file_name"]).name
        candidates = lookup.get(name, [])

        if len(candidates) != 1:
            issues.append({
                "split": split,
                "filename": name,
                "issue": "missing_or_ambiguous_image"
            })
            continue

        with Image.open(candidates[0]) as image:
            actual_width, actual_height = image.size
            orientation = image.getexif().get(274, 1)

        checked += 1
        expected_width = record["width"]
        expected_height = record["height"]

        if (
            actual_width != expected_width
            or actual_height != expected_height
        ):
            issues.append({
                "split": split,
                "filename": name,
                "issue": "dimension_mismatch",
                "image_path": str(candidates[0]),
                "json_width": expected_width,
                "json_height": expected_height,
                "actual_width": actual_width,
                "actual_height": actual_height,
                "exif_orientation": orientation,
                "width_height_swapped": (
                    actual_width == expected_height
                    and actual_height == expected_width
                )
            })

report = pd.DataFrame(issues)
report.to_csv(audit / "ruod_dimension_issues.csv", index=False)

print("Images checked:", checked)
print("Issue entries:", len(report))

if len(report):
    display(report.head(20))
else:
    print("No dimension mismatches found.")

print("Saved:", audit / "ruod_dimension_issues.csv")

Images checked: 13704
Issue entries: 297


,split,filename,issue,image_path,json_width,json_height,actual_width,actual_height,exif_orientation,width_height_swapped
0,train,008431.jpg,dimension_mismatch,/kaggle/working/ruod_original/RUOD/RUOD_pic/tr...,1024.0,683.0,683.0,1024.0,6.0,True
1,test,001464.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,test,001102.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,test,012390.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,test,000005.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,test,004650.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,test,003473.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,test,010399.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,test,004443.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,test,001475.jpg,missing_or_ambiguous_image,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved: /kaggle/working/three_domain_audit/ruod_dimension_issues.csv


## Resolve Duplicate Paths and Check Image Orientation

In [11]:
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageOps
import json
import pandas as pd

source = Path("/kaggle/working/ruod_original")
audit = Path("/kaggle/working/three_domain_audit")

lookup = defaultdict(list)

for path in source.rglob("*"):
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup[path.name].append(path)

resolved_rows = []
unresolved = []
duplicate_examples = []

for split in ["train", "test"]:
    json_files = [
        path for path in source.rglob(f"instances_{split}.json")
        if path.parent.name == "RUOD_ANN"
    ]
    if len(json_files) != 1:
        raise RuntimeError(f"Cannot locate {split} JSON uniquely")

    data = json.loads(json_files[0].read_text())

    for record in data["images"]:
        name = Path(record["file_name"]).name
        candidates = lookup.get(name, [])

        if len(candidates) > 1 and len(duplicate_examples) < 3:
            duplicate_examples.append({
                "filename": name,
                "paths": [str(path) for path in candidates]
            })

        # Prefer the original split inside the main RUOD_pic folder.
        preferred = [
            path for path in candidates
            if "RUOD_pic" in path.parts
            and split in path.relative_to(source).parts[:-1]
        ]

        if len(preferred) == 1:
            chosen = preferred[0]
        elif len(candidates) == 1:
            chosen = candidates[0]
        else:
            unresolved.append({
                "split": split,
                "filename": name,
                "issue": "path_not_resolved",
                "candidate_paths": [str(path) for path in candidates]
            })
            continue

        with Image.open(chosen) as image:
            raw_size = image.size
            orientation = image.getexif().get(274, 1)
            oriented_size = ImageOps.exif_transpose(image).size

        expected = (record["width"], record["height"])

        if oriented_size != expected:
            unresolved.append({
                "split": split,
                "filename": name,
                "issue": "dimensions_still_mismatch",
                "raw_size": raw_size,
                "oriented_size": oriented_size,
                "expected_size": expected
            })
            continue

        resolved_rows.append({
            "original_split": split,
            "image_id": record["id"],
            "filename": name,
            "original_path": str(chosen),
            "width": expected[0],
            "height": expected[1],
            "exif_orientation": orientation,
            "needs_exif_normalization": orientation != 1
        })

resolved = pd.DataFrame(resolved_rows)
resolved.to_csv(audit / "ruod_resolved_images.csv", index=False)

pd.DataFrame(unresolved).to_csv(
    audit / "ruod_unresolved_images.csv", index=False
)

print("Resolved images:", len(resolved))
print("Unresolved images:", len(unresolved))
print(
    "Images needing EXIF normalization:",
    int(resolved["needs_exif_normalization"].sum())
)

print("\nDuplicate filename examples:")
for example in duplicate_examples:
    print(example["filename"])
    for path in example["paths"]:
        print(" ", path)

if unresolved:
    print("\nUnresolved examples:")
    display(pd.DataFrame(unresolved).head(10))

Resolved images: 14000
Unresolved images: 0
Images needing EXIF normalization: 11

Duplicate filename examples:
001464.jpg
  /kaggle/working/ruod_original/RUOD/RUOD_pic/test/001464.jpg
  /kaggle/working/ruod_original/RUOD/Environment_pic/light/001464.jpg
001102.jpg
  /kaggle/working/ruod_original/RUOD/RUOD_pic/test/001102.jpg
  /kaggle/working/ruod_original/RUOD/Environment_pic/blur/001102.jpg
012390.jpg
  /kaggle/working/ruod_original/RUOD/RUOD_pic/test/012390.jpg
  /kaggle/working/ruod_original/RUOD/Environment_pic/blur/012390.jpg


## Verify Boxes on the 11 EXIF-Oriented Images

## preparation

In [12]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import json
import math

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

all_records = []
corrections = []
exclusions = []

def export_image(domain, key, source_path, width, height,
                 boxes, group, original_split="unknown"):
    """Export one image and its fish-only labels."""
    source_path = Path(source_path)

    if not source_path.exists():
        raise FileNotFoundError(source_path)

    folder = BASE / domain
    images = folder / "images"
    labels = folder / "labels"
    images.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)

    destination = images / f"{key}{source_path.suffix.lower()}"
    label_path = labels / f"{key}.txt"

    lines = []

    for box in boxes:
        x, y, w, h = map(float, box)

        if (
            not all(math.isfinite(v) for v in [x, y, w, h])
            or w <= 0 or h <= 0
        ):
            raise ValueError(f"Invalid box: {domain}/{key}: {box}")

        x1, y1 = max(0.0, x), max(0.0, y)
        x2 = min(float(width), x + w)
        y2 = min(float(height), y + h)

        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"Outside box: {domain}/{key}: {box}")

        clipped = [x1, y1, x2-x1, y2-y1]

        if not np.allclose(box, clipped, atol=1e-6, rtol=0):
            corrections.append({
                "domain": domain, "image": key,
                "original": list(box), "clipped": clipped
            })

        lines.append(
            f"0 {(x1+x2)/(2*width):.8f} "
            f"{(y1+y2)/(2*height):.8f} "
            f"{(x2-x1)/width:.8f} {(y2-y1)/height:.8f}"
        )

    # Links avoid duplicating several GB in this session.
    if destination.is_symlink():
        if destination.resolve() != source_path.resolve():
            raise RuntimeError(f"Wrong existing link: {destination}")
    elif destination.exists():
        raise RuntimeError(f"Unexpected existing file: {destination}")
    else:
        destination.symlink_to(source_path.resolve())

    label_path.write_text("\n".join(lines) + ("\n" if lines else ""))

    all_records.append({
        "domain": domain,
        "key": key,
        "image_path": str(destination),
        "label_path": str(label_path),
        "source_path": str(source_path),
        "width": width,
        "height": height,
        "fish_boxes": len(lines),
        "group": str(group),
        "original_split": original_split
    })

# Use the NorFisk manifest already verified in Cell 9.
nf = pd.read_csv(
    "/kaggle/working/three_domain_audit/norfisk_manifest.csv"
)

if not nf["status"].eq("ok").all():
    raise RuntimeError("NorFisk manifest contains unresolved issues.")

for row in nf.itertuples(index=False):
    path = Path(row.image_path)
    export_image(
        domain="norfisk",
        key=path.stem,
        source_path=path,
        width=row.width,
        height=row.height,
        boxes=[[
            row.xmin, row.ymin,
            row.xmax-row.xmin, row.ymax-row.ymin
        ]],
        # Individual image identity; not a recording ID.
        group=path.stem
    )

print("NorFisk exported:", len(nf), "images")

NorFisk exported: 12514 images


In [13]:
from pathlib import Path
import shutil
import pandas as pd

usage = shutil.disk_usage("/kaggle/working")
print(f"Total storage: {usage.total / 2**30:.2f} GB")
print(f"Used storage:  {usage.used / 2**30:.2f} GB")
print(f"Free storage:  {usage.free / 2**30:.2f} GB")

archive_dir = Path("/kaggle/working/ruod_download")
rows = []

if archive_dir.exists():
    for p in sorted(archive_dir.iterdir()):
        if p.is_file():
            rows.append({
                "file": str(p),
                "size_GB": round(p.stat().st_size / 2**30, 3)
            })

display(pd.DataFrame(rows))

frame_dir = Path("/kaggle/working/lodo_ready/brackish_frames")
print(
    "Brackish PNG files currently present:",
    sum(1 for _ in frame_dir.glob("*.png"))
)

Total storage: 19.52 GB
Used storage:  15.48 GB
Free storage:  4.02 GB


,file,size_GB
0,/kaggle/working/ruod_download/RUOD.tar,3.458
1,/kaggle/working/ruod_download/RUOD.tar.partaa,1.953
2,/kaggle/working/ruod_download/RUOD.tar.partab,1.505


Brackish PNG files currently present: 0


In [14]:
from pathlib import Path
import shutil

# Confirm that extracted RUOD data is present.
ruod_root = Path("/kaggle/working/ruod_original/RUOD")

jpg_count = sum(1 for _ in ruod_root.rglob("*.jpg"))
main_annotations = list(ruod_root.rglob("instances_train.json"))
test_annotations = list(ruod_root.rglob("instances_test.json"))

assert jpg_count >= 14000, f"Unexpected RUOD image count: {jpg_count}"
assert main_annotations and test_annotations, "RUOD annotations missing."

# Remove only redundant downloaded archive files.
archive_dir = Path("/kaggle/working/ruod_download")

for name in ["RUOD.tar", "RUOD.tar.partaa", "RUOD.tar.partab"]:
    p = archive_dir / name
    if p.is_file():
        size = p.stat().st_size / 2**30
        p.unlink()
        print(f"Removed: {name} ({size:.2f} GB)")

# Remove the PNG whose write failed, so extraction recreates it.
failed_frame = Path(
    "/kaggle/working/lodo_ready/brackish_frames/"
    "2019-03-21_03-21-54to2019-03-21_03-22-09_1-0120.png"
)
failed_frame.unlink(missing_ok=True)

print(
    f"\nFree storage: "
    f"{shutil.disk_usage('/kaggle/working').free / 2**30:.2f} GB"
)

Removed: RUOD.tar (3.46 GB)
Removed: RUOD.tar.partaa (1.95 GB)
Removed: RUOD.tar.partab (1.51 GB)

Free storage: 10.94 GB


In [18]:
from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

manifest_path = AUDIT / "all_images.csv"

print("Manifest exists:", manifest_path.is_file())
print("Export function available:", callable(globals().get("export_image")))
print("In-memory records:", len(globals().get("all_records", [])))

for domain in ["brackish", "ruod", "norfisk"]:
    image_dir = BASE / domain / "images"
    label_dir = BASE / domain / "labels"

    print(
        domain,
        "| readable images:", sum(p.is_file() for p in image_dir.glob("*")),
        "| labels:", sum(p.is_file() for p in label_dir.glob("*.txt")),
    )

if manifest_path.is_file():
    manifest = pd.read_csv(manifest_path)
    print("\nManifest columns:", manifest.columns.tolist())
    print("Duplicate keys:", manifest.duplicated(["domain", "key"]).sum())
    display(manifest.groupby("domain").size().rename("images"))

Manifest exists: False
Export function available: True
In-memory records: 12514
brackish | readable images: 0 | labels: 0
ruod | readable images: 0 | labels: 0
norfisk | readable images: 12514 | labels: 12514


In [19]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import cv2
import json
import pandas as pd
import shutil

BASE = Path("/kaggle/working/lodo_ready")
AUDIT = BASE / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)

# Keep the already exported NorFisk records.
# This also makes rerunning this cell safe for the in-memory manifest.
all_records = [
    r for r in all_records if r["domain"] == "norfisk"
]
assert len(all_records) == 12514, "Run the NorFisk export cell first."

# ---------- RUOD ----------
ruod_source = Path("/kaggle/working/ruod_original")
resolved_path = Path(
    "/kaggle/working/three_domain_audit/ruod_resolved_images.csv"
)
assert resolved_path.is_file(), "Run the RUOD path-resolution cell first."

resolved = pd.read_csv(resolved_path)
lookup = {
    (r.original_split, r.filename): r
    for r in resolved.itertuples(index=False)
}

for split in ["train", "test"]:
    matches = [
        p for p in ruod_source.rglob(f"instances_{split}.json")
        if p.parent.name == "RUOD_ANN"
    ]
    assert len(matches) == 1, f"Cannot locate RUOD {split} annotations."

    data = json.loads(matches[0].read_text())
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() == "fish"
    }

    boxes = defaultdict(list)
    for ann in data["annotations"]:
        if ann["category_id"] in fish_ids:
            boxes[ann["image_id"]].append(ann["bbox"])

    for record in data["images"]:
        filename = Path(record["file_name"]).name

        # Previously identified orientation/box-alignment issue.
        if filename == "008431.jpg":
            exclusions.append({
                "domain": "ruod",
                "image": filename,
                "reason": "Unresolved orientation/annotation alignment",
            })
            continue

        row = lookup[(split, filename)]
        orientation = int(row.exif_orientation)
        assert orientation in [0, 1], (
            f"Review orientation: {filename}, EXIF={orientation}"
        )

        source = Path(row.original_path)
        with Image.open(source) as image:
            actual_size = image.size

        assert actual_size == (record["width"], record["height"]), (
            f"Dimension mismatch: {filename}"
        )

        key = f"{split}_{Path(filename).stem}"
        export_image(
            domain="ruod",
            key=key,
            source_path=source,
            width=record["width"],
            height=record["height"],
            boxes=boxes[record["id"]],
            group=key,
            original_split=split,
        )

    print("RUOD exported:", split, flush=True)

# ---------- BRACKISH ----------
matches = [
    p for p in Path("/kaggle/input").rglob("train_groundtruth.json")
    if "brackish" in str(p).lower()
]
assert len(matches) == 1, "Add the Brackish dataset to notebook inputs."

annotation_dir = matches[0].parent
brackish_root = annotation_dir.parent.parent

records_by_name = {}
fish_boxes = defaultdict(list)

for split in ["train", "valid", "test"]:
    data = json.loads(
        (annotation_dir / f"{split}_groundtruth.json").read_text()
    )
    fish_ids = {
        c["id"] for c in data["categories"]
        if c["name"].strip().lower() in {"fish", "small_fish"}
    }
    id_to_name = {}

    for record in data["images"]:
        filename = Path(record["file_name"]).name
        assert filename not in records_by_name, (
            f"Repeated Brackish image record: {filename}"
        )
        records_by_name[filename] = (record, split)
        id_to_name[record["id"]] = filename

    for ann in data["annotations"]:
        if ann["category_id"] in fish_ids:
            fish_boxes[id_to_name[ann["image_id"]]].append(ann["bbox"])

videos = defaultdict(list)
for p in brackish_root.rglob("*.avi"):
    videos[p.stem].append(p)

frames_by_video = defaultdict(dict)
for filename in records_by_name:
    video_id, number = Path(filename).stem.rsplit("-", 1)
    frames_by_video[video_id][int(number)] = filename

frame_dir = BASE / "brackish_frames"
frame_dir.mkdir(parents=True, exist_ok=True)

for index, (video_id, wanted) in enumerate(
    sorted(frames_by_video.items()), start=1
):
    assert len(videos[video_id]) == 1, f"Video missing/ambiguous: {video_id}"

    pending = {}
    for number, filename in wanted.items():
        destination = frame_dir / filename

        if destination.exists():
            existing = cv2.imread(str(destination))
            if existing is not None and existing.shape[:2] == (540, 960):
                continue
            destination.unlink()

        pending[number] = filename

    if pending:
        capture = cv2.VideoCapture(str(videos[video_id][0]))
        assert capture.isOpened(), f"Cannot open video: {video_id}"
        number = 0

        try:
            while pending:
                ok, frame = capture.read()
                if not ok:
                    break
                number += 1

                if number not in pending:
                    continue

                resized = cv2.resize(
                    frame, (960, 540), interpolation=cv2.INTER_CUBIC
                )
                destination = frame_dir / pending[number]

                if not cv2.imwrite(str(destination), resized):
                    destination.unlink(missing_ok=True)
                    raise RuntimeError(
                        "Frame write failed. Free storage: "
                        f"{shutil.disk_usage(BASE).free / 2**30:.2f} GB"
                    )

                del pending[number]
        finally:
            capture.release()

        assert not pending, f"Missing frames in {video_id}: {len(pending)}"

    for filename in wanted.values():
        record, split = records_by_name[filename]
        assert (record["width"], record["height"]) == (960, 540)

        export_image(
            domain="brackish",
            key=Path(filename).stem,
            source_path=frame_dir / filename,
            width=960,
            height=540,
            boxes=fish_boxes[filename],
            group=video_id[:10],  # Recording date
            original_split=split,
        )

    if index % 10 == 0 or index == len(frames_by_video):
        print(
            f"Brackish videos: {index}/{len(frames_by_video)}",
            flush=True,
        )

# ---------- SAVE MANIFEST AND LOGS ----------
manifest = pd.DataFrame(all_records)
assert not manifest.duplicated(["domain", "key"]).any()

assert manifest.groupby("domain").size().to_dict() == {
    "brackish": 12444,
    "norfisk": 12514,
    "ruod": 13999,
}

manifest.to_csv(AUDIT / "all_images.csv", index=False)

def save_log(records, filename):
    unique = {}
    for record in records:
        signature = json.dumps(record, sort_keys=True, default=str)
        unique[signature] = record
    pd.DataFrame(unique.values()).to_csv(AUDIT / filename, index=False)

save_log(corrections, "box_corrections.csv")
save_log(exclusions, "excluded_images.csv")

display(
    manifest.groupby("domain").agg(
        images=("key", "size"),
        fish_boxes=("fish_boxes", "sum"),
    )
)
print("EXPORT COMPLETE — manifest saved.")
print(
    "Free storage:",
    round(shutil.disk_usage(BASE).free / 2**30, 2),
    "GB",
)

RUOD exported: train
RUOD exported: test
Brackish videos: 10/89
Brackish videos: 20/89
Brackish videos: 30/89
Brackish videos: 40/89
Brackish videos: 50/89
Brackish videos: 60/89
Brackish videos: 70/89
Brackish videos: 80/89
Brackish videos: 89/89


,images,fish_boxes
domain,,
brackish,12444,14120
norfisk,12514,12514
ruod,13999,13091


EXPORT COMPLETE — manifest saved.
Free storage: 6.5 GB


In [20]:
import hashlib
from sklearn.model_selection import GroupShuffleSplit
import yaml

manifest = pd.read_csv(AUDIT / "all_images.csv")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = []
for index, path in enumerate(manifest["source_path"], start=1):
    hashes.append(sha256_file(path))
    if index % 5000 == 0:
        print("Files hashed:", index)

manifest["sha256"] = hashes

cross_domain = (
    manifest.groupby("sha256")["domain"].nunique()
)
overlap_hashes = cross_domain[cross_domain > 1].index

manifest.to_csv(AUDIT / "all_images_hashed.csv", index=False)

if len(overlap_hashes):
    manifest[manifest["sha256"].isin(overlap_hashes)].to_csv(
        AUDIT / "cross_domain_duplicates.csv", index=False
    )
    raise RuntimeError(
        "Cross-domain duplicates found. Review before training."
    )

manifest["source_partition"] = ""

for domain in ["brackish", "ruod", "norfisk"]:
    domain_rows = manifest[manifest["domain"] == domain]

    if domain == "brackish":
        # Union date groups connected by any exact duplicate.
        parent = {group: group for group in domain_rows["group"].unique()}

        def find(group):
            while parent[group] != group:
                parent[group] = parent[parent[group]]
                group = parent[group]
            return group

        for _, duplicates in domain_rows.groupby("sha256"):
            groups = duplicates["group"].unique()
            for group in groups[1:]:
                parent[find(group)] = find(groups[0])

        split_groups = domain_rows["group"].map(find).to_numpy()
    else:
        split_groups = domain_rows["sha256"].to_numpy()

    splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=42
    )
    train_index, val_index = next(
        splitter.split(domain_rows, groups=split_groups)
    )

    manifest.loc[
        domain_rows.index[train_index], "source_partition"
    ] = "train"
    manifest.loc[
        domain_rows.index[val_index], "source_partition"
    ] = "val"

    train_hashes = set(domain_rows.iloc[train_index]["sha256"])
    val_hashes = set(domain_rows.iloc[val_index]["sha256"])
    assert train_hashes.isdisjoint(val_hashes)

manifest.to_csv(AUDIT / "fixed_partitions.csv", index=False)

config_dir = BASE / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

runs = [
    ("brackish_ruod_to_norfisk", ["brackish", "ruod"], "norfisk"),
    ("ruod_norfisk_to_brackish", ["ruod", "norfisk"], "brackish"),
    ("norfisk_brackish_to_ruod", ["norfisk", "brackish"], "ruod")
]

for name, sources, target in runs:
    train = manifest[
        manifest["domain"].isin(sources)
        & manifest["source_partition"].eq("train")
    ]
    val = manifest[
        manifest["domain"].isin(sources)
        & manifest["source_partition"].eq("val")
    ]
    test = manifest[manifest["domain"].eq(target)]

    lists = {}
    for split, table in [("train", train), ("val", val), ("test", test)]:
        path = config_dir / f"{name}_{split}.txt"
        path.write_text("\n".join(table["image_path"]) + "\n")
        lists[split] = str(path)

    config = {
        "path": str(BASE),
        **lists,
        "names": {0: "fish"}
    }

    (config_dir / f"{name}.yaml").write_text(
        yaml.safe_dump(config, sort_keys=False)
    )

    print(
        f"{name}: train={len(train)}, "
        f"source_val={len(val)}, unseen_test={len(test)}"
    )

display(
    manifest.groupby(["domain", "source_partition"]).agg(
        images=("key", "size"),
        fish_boxes=("fish_boxes", "sum")
    )
)

print("\nConfigs saved:", config_dir)
print("PREPARATION COMPLETE — no model trained yet.")

Files hashed: 5000
Files hashed: 10000
Files hashed: 15000
Files hashed: 20000
Files hashed: 25000
Files hashed: 30000
Files hashed: 35000
brackish_ruod_to_norfisk: train=20474, source_val=5969, unseen_test=12514
ruod_norfisk_to_brackish: train=21230, source_val=5283, unseen_test=12444
norfisk_brackish_to_ruod: train=19266, source_val=5692, unseen_test=13999


images  fish_boxes
domain   source_partition                    
brackish train               9255       10255
         val                 3189        3865
norfisk  train              10011       10011
         val                 2503        2503
ruod     train              11219       10655
         val                 2780        2436


Configs saved: /kaggle/working/lodo_ready/configs
PREPARATION COMPLETE — no model trained yet.


##  Training

In [21]:
%pip install -q ultralytics==8.4.207

Note: you may need to restart the kernel to use updated packages.


In [22]:
from pathlib import Path
import torch
from ultralytics import YOLO

BASE = Path("/kaggle/working/lodo_ready")
RUN_NAME = "brackish_ruod_to_norfisk"
CONFIG = BASE / "configs" / f"{RUN_NAME}.yaml"

assert CONFIG.exists(), f"Config missing: {CONFIG}"
assert torch.cuda.is_available(), "Enable GPU before training."

print("GPU:", torch.cuda.get_device_name(0))
print("Training: Brackish + RUOD")
print("Validation: source domains only")
print("NorFisk remains unused during training.")

# Fresh COCO-pretrained initialization.
# Do not use the previous Brackish pilot's best.pt.
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(CONFIG),
    project="/kaggle/working/lodo_runs",
    name=RUN_NAME,
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    cache=False,
    seed=42,
    deterministic=True,
    patience=30,
    save=True,
    save_period=5,
    plots=True,
    exist_ok=False,
)

print("\nTraining finished.")
print("Results folder:", model.trainer.save_dir)
print("Best weights:", model.trainer.best)

KeyboardInterrupt: 